# Visual review of candidate Sentinel imagery

Create fixed-display comparisons from ten small retained GeoTIFF chips. Inspect imagery around current OSM outlines and a separate historical municipal floodplain reference. These previews do not classify flooding or building damage.

Default execution is offline using cached chips; fresh downloads require Earth Engine authentication and explicitly enabling the remote cell. Downloaded chips total under 1 MB at the original review. Inspect the rendered figures after every rendering change. Reruns replace figures/diagnostics and record a new run.

In [ ]:
from pathlib import Path
import json

# In Colab, clone the repository and change into it before running this cell.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/ethekwini.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the africa-map checkout or one of its subdirectories.")
CONFIG = json.loads((ROOT / "config/ethekwini.yaml").read_text())
print("Project:", CONFIG["project_id"])


In [ ]:
RUN_REMOTE_DOWNLOADS = False  # Enable to obtain a new registered chip snapshot.
if RUN_REMOTE_DOWNLOADS:
    from pathlib import Path
    from datetime import datetime,timezone
    from urllib.request import urlopen
    import json
    import ee
    import geopandas as gpd
    import pandas as pd
    root=ROOT;cfg=CONFIG
    ee.Initialize(project=cfg['availability']['earth_engine_project'])
    source=root/'data/raw/earth_engine_visual_review';source.mkdir(parents=True,exist_ok=True)
    sites=gpd.read_file(root/'data/processed/ethekwini/flagged_site_review.gpkg',layer='osm_outlines').to_crs(cfg['analysis_crs'])
    assets=pd.read_csv(root/'data/processed/ethekwini/earth_engine_asset_matches.csv')
    assets=assets.loc[assets.sensor.eq('S1') | assets.query_date.isin(['2022-03-29','2022-04-13','2022-04-28'])]
    previous=json.loads((source/'download_manifest.json').read_text()) if (source/'download_manifest.json').exists() else []
    manifest=[]
    for site in sites.itertuples():
     bounds=site.geometry.buffer(500).bounds
     region=ee.Geometry.Rectangle(list(bounds),proj=cfg['analysis_crs'],geodesic=False)
     for asset in assets.itertuples():
      bands=['VV','VH','angle'] if asset.sensor=='S1' else ['B2','B3','B4','B8','B11','SCL']
      scale=10 if asset.sensor=='S1' else 20
      image=ee.Image(asset.earth_engine_asset_id).select(bands).toFloat()
      name=f'{site.institution_id.split("/")[-1]}_{asset.sensor}_{asset.query_date}.tif'
      path=source/name
      if not path.exists():
       url=image.getDownloadURL({'region':region,'crs':cfg['analysis_crs'],'scale':scale,'format':'GEO_TIFF','filePerBand':False})
       with urlopen(url,timeout=60) as response:data=response.read()
       path.write_bytes(data)
      manifest.append({'file':name,'institution_id':site.institution_id,'sensor':asset.sensor,'acquisition_date':asset.query_date,'earth_engine_asset_id':asset.earth_engine_asset_id,'bands':bands,'scale_m':scale,'crs':cfg['analysis_crs'],'region_bounds_projected':list(bounds),'region_definition':'bounds of current OSM outline buffered by 500 m in EPSG:32736','downloaded_at_utc':next((r['downloaded_at_utc'] for r in previous if r['file']==name),datetime.now(timezone.utc).isoformat())})
      (source/'download_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
      print(name,path.stat().st_size,flush=True)
else:
    print("Using retained small GeoTIFF chips; no remote calls.")


In [ ]:
assert CONFIG['visual_review'] == {'source_directory':'data/raw/earth_engine_visual_review','context_buffer_m':500,'sar_scale_m':10,'optical_scale_m':20,'rgb_sr_min':0,'rgb_sr_max':0.30,'rgb_gamma':1.2,'vv_display_min_db':-25,'vv_display_max_db':0,'vh_display_min_db':-30,'vh_display_max_db':-5}, 'Review rendering code before changing display settings.'
from pathlib import Path
import os
os.environ.setdefault('MPLCONFIGDIR','/tmp/africa-map-matplotlib')
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

ROOT=ROOT
raw=ROOT/'data/raw/earth_engine_visual_review';output=ROOT/'outputs/figures';output.mkdir(parents=True,exist_ok=True)
sites=gpd.read_file(ROOT/'data/processed/ethekwini/flagged_site_review.gpkg',layer='osm_outlines').to_crs(32736)
flood=gpd.read_file(ROOT/'data/processed/ethekwini/floodplain_screening.gpkg',layer='mapped_floodplain').to_crs(32736)
records=[]
for site in sites.itertuples():
 id=site.institution_id.split('/')[-1]
 name='Unnamed library — candidate Amanzimtoti Library' if id=='368501740' else 'Umkhumbane Museum'
 fig,axes=plt.subplots(2,4,figsize=(16,9),layout='constrained')
 panels=[('S2','2022-03-29','RGB'),('S2','2022-04-13','RGB'),('S2','2022-04-28','RGB'),(None,None,None),('S1','2022-04-05','VV'),('S1','2022-04-17','VV'),('S1','2022-04-05','VH'),('S1','2022-04-17','VH')]
 for ax,(sensor,date,display) in zip(axes.flat,panels):
  if sensor is None:
   bounds=site.geometry.buffer(500).bounds
   flood.loc[flood.intersects(site.geometry.buffer(800))].plot(ax=ax,facecolor='#ffedd5',edgecolor='#c2410c',linewidth=.8)
   gpd.GeoSeries([site.geometry],crs=32736).plot(ax=ax,facecolor='none',edgecolor='#0891b2',linewidth=1.6)
   ax.set_xlim(bounds[0],bounds[2]);ax.set_ylim(bounds[1],bounds[3]);ax.set_title('Reference: historical municipal floodplain\nOrange = mapped polygon; cyan = OSM outline',fontsize=9)
  else:
   with rasterio.open(raw/f'{id}_{sensor}_{date}.tif') as src:
    data=src.read(masked=True);extent=[src.bounds.left,src.bounds.right,src.bounds.bottom,src.bounds.top]
    assert src.crs.to_epsg()==32736
    footprint_mask=geometry_mask([site.geometry.__geo_interface__],out_shape=(src.height,src.width),transform=src.transform,invert=True)
    context_mask=geometry_mask([site.geometry.buffer(100).__geo_interface__],out_shape=(src.height,src.width),transform=src.transform,invert=True)
    if display=='RGB':
     rgb=np.moveaxis(data[[2,1,0]].filled(np.nan),0,-1)
     rgb=np.clip(rgb/3000,0,1)**(1/1.2)
     ax.imshow(rgb,extent=extent,origin='upper',interpolation='nearest')
     retained=np.isin(data[5].filled(0),[4,5,6])
     records.append({'institution_id':site.institution_id,'sensor':sensor,'date':date,'metric':'optical_outline_center_samples','value':int(footprint_mask.sum()),'units':'pixel centers','notes':'20 m display sampling; boundary area not counted; not independent damage observations'})
     title=f'Sentinel-2 RGB · {date}\n'+('Cloud-obscured; no ground inference' if date=='2022-04-13' else '20 m grid; SR stretch 0–0.30, gamma 1.2')
    else:
     index=0 if display=='VV' else 1;lo,hi=(-25,0) if display=='VV' else (-30,-5)
     im=ax.imshow(data[index],extent=extent,origin='upper',cmap='gray',vmin=lo,vmax=hi,interpolation='nearest')
     title=f'Sentinel-1 {display} · {date}\nUnfiltered GRD; fixed {lo} to {hi} dB'
     if display=='VV':
      for metric,values,units in [('context_incidence_angle_median',data[2][context_mask],'degrees'),('context_vv_median',data[0][context_mask],'dB'),('context_vh_median',data[1][context_mask],'dB')]:
       records.append({'institution_id':site.institution_id,'sensor':sensor,'date':date,'metric':metric,'value':float(np.ma.median(values)),'units':units,'notes':'Planar 100 m context, local chip pixel centers; descriptive QA only'})
      records.append({'institution_id':site.institution_id,'sensor':sensor,'date':date,'metric':'sar_outline_center_samples','value':int(footprint_mask.sum()),'units':'pixel centers','notes':'10 m sampling; boundary area not counted; not effective resolution'})
    gpd.GeoSeries([site.geometry],crs=32736).boundary.plot(ax=ax,color='#00ffff',linewidth=1)
    ax.set_xlim(extent[:2]);ax.set_ylim(extent[2:]);ax.set_title(title,fontsize=9)
  ax.set_aspect('equal');ax.set_xticks([]);ax.set_yticks([]);ax.set_xlabel('');ax.set_ylabel('')
  x0,x1=ax.get_xlim();y0,y1=ax.get_ylim()
  ax.plot([x0+50,x0+250],[y0+50,y0+50],color='#e11d48',linewidth=2)
  ax.text(x0+150,y0+65,'200 m',ha='center',fontsize=8,color='#e11d48',bbox={'facecolor':'white','alpha':.8,'edgecolor':'none','pad':1})
  ax.text(.95,.94,'N ↑',ha='right',va='top',transform=ax.transAxes,fontsize=9,color='#e11d48',bbox={'facecolor':'white','alpha':.8,'edgecolor':'none','pad':1})
 fig.suptitle(name+'\nVisual feasibility review — current OSM outline; no observed-flood or damage classification',fontsize=13)
 fig.supxlabel('Sentinel imagery: EU/ESA/Copernicus via Earth Engine · OSM outlines: © OpenStreetMap contributors · Municipal floodplain: historical mapping\nCyan outlines are current mapped buildings, not verified April 2022 geometry. Display pixels do not establish ground visibility or building condition.',fontsize=8)
 path=output/f'visual_review_{id}.png';fig.savefig(path,dpi=150);plt.close(fig)
 print(path.relative_to(ROOT),flush=True)
pd.DataFrame(records).to_csv(ROOT/'data/processed/ethekwini/visual_review_diagnostics.csv',index=False)
print('Diagnostic table written.')


In [ ]:
from datetime import datetime, timezone
import csv
import hashlib
import platform
import shutil
import subprocess

started = datetime.now(timezone.utc)
run_id = "visual_review_" + started.strftime("%Y%m%dT%H%M%S%fZ")
raw = ROOT / CONFIG["visual_review"]["source_directory"]
manifest_path = raw / "download_manifest.json"
manifest = json.loads(manifest_path.read_text())
assert len(manifest) == 10
for item in manifest:
    with rasterio.open(raw / item["file"]) as image:
        assert image.crs.to_epsg() == 32736
        assert image.count == len(item["bands"])
        assert abs(image.res[0] - item["scale_m"]) < 1e-6
        assert image.width < 200 and image.height < 200
source_files = [manifest_path] + [raw / item["file"] for item in manifest]
source_checksums = ROOT / "metadata/earth_engine_visual_source_checksums.json"
source_checksums.write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}, indent=2) + "\n")
summary = {
    "run_id": run_id, "chip_count": len(manifest), "total_chip_bytes": sum((raw/r["file"]).stat().st_size for r in manifest),
    "context_buffer_m": 500, "optical_display_scale_m": 20, "sar_display_scale_m": 10,
    "display": CONFIG["visual_review"],
    "visual_review_status": "Reviewed rendered comparisons for both sites; see research/visual_review.md for evidence-qualified observations",
    "building_flood_labels_assigned": False, "flood_detection_performed": False,
    "recommendation": "An exploratory SAR change comparison is technically possible, but no building flood labels are justified. Use wider spatial context and independent dated evidence; peak inundation remains unobserved.",
    "limitations": "Unfiltered SAR texture and urban scattering; very few building samples; current OSM outlines; historical floodplain; optical post-event date is late. Visual absence is not a negative flood label.",
}
summary_path = ROOT / "metadata/visual_review_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
run_directory = ROOT / "metadata/runs" / run_id
run_directory.mkdir(parents=True, exist_ok=False)
(run_directory / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n")
env = {"python": platform.python_version(), "rasterio": rasterio.__version__, "matplotlib": matplotlib.__version__, "numpy": np.__version__, "geopandas": gpd.__version__}
(run_directory / "environment.json").write_text(json.dumps(env, indent=2) + "\n")
inputs = source_files + [ROOT / "data/processed/ethekwini/flagged_site_review.gpkg", ROOT / "data/processed/ethekwini/floodplain_screening.gpkg", ROOT / "data/processed/ethekwini/earth_engine_asset_matches.csv"]
(run_directory / "input_checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs}, indent=2) + "\n")
code = ROOT / "notebooks/05_visual_review.ipynb"
shutil.copyfile(code, run_directory / code.name)
(run_directory / "code_checksums.json").write_text(json.dumps({str(code.relative_to(ROOT)): hashlib.sha256(code.read_bytes()).hexdigest()}, indent=2) + "\n")
outputs = [ROOT / f"outputs/figures/visual_review_{way}.png" for way in CONFIG["site_review"]["way_ids"]] + [ROOT / "data/processed/ethekwini/visual_review_diagnostics.csv", summary_path, source_checksums]
(run_directory / "checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in outputs}, indent=2) + "\n")
dataset_id = "earth_engine_visual_chips_20261020"
source_row = {"dataset_id": dataset_id, "title": "Small Sentinel visual-review GeoTIFF chips at two eThekwini sites",
    "provider": "EU/ESA/Copernicus via Google Earth Engine", "version": "2026-10-20 small-region downloads; exact assets/bands/grid retained",
    "source_url_or_asset_id": "COPERNICUS/S1_GRD;COPERNICUS/S2_SR_HARMONIZED", "status": "accessed",
    "accessed_at_utc": min(r["downloaded_at_utc"] for r in manifest), "observation_start": "2022-03-29", "observation_end": "2022-04-28",
    "geographic_coverage": "500 m buffered outline bounding boxes around two flagged sites", "resolution_or_scale": "10 m SAR chip / 20 m optical chip sampling; RGB is rendered at 20 m",
    "crs": "EPSG:32736", "license": "Copernicus Sentinel data terms; OSM and municipal reference layers retain their own source conditions",
    "redistribution_conditions": "Local review only; source chips and figures not published to website",
    "limitations": summary["limitations"] + " Register sha256 hashes the source-file checksum manifest.",
    "storage_location": str(source_checksums.relative_to(ROOT)), "sha256": hashlib.sha256(source_checksums.read_bytes()).hexdigest()}
run_row = {"run_id": run_id, "started_at_utc": started.isoformat(),
    "input_dataset_ids": ";".join([dataset_id,"earth_engine_quality_snapshot_20261020","osm_flagged_sites_snapshot_20261020",CONFIG["floodplain_dataset_id"]]),
    "scene_manifest_path": str(manifest_path.relative_to(ROOT)),
    "code_commit": subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip(),
    "working_tree_dirty": str(bool(subprocess.check_output(["git","status","--porcelain"],cwd=ROOT,text=True).strip())).lower(),
    "config_snapshot_path": str((run_directory / "config.json").relative_to(ROOT)), "environment_path": str((run_directory / "environment.json").relative_to(ROOT)),
    "method": "Fixed-stretch optical/SAR comparisons; current outline overlays; historical floodplain reference; pixel-center and descriptive incidence/backscatter diagnostics",
    "output_locations": ";".join(str(p.relative_to(ROOT)) for p in outputs), "output_checksums_path": str((run_directory / "checksums.json").relative_to(ROOT)),
    "validation_status": "partial", "notes": "Figures visually inspected. No flood labels, speckle filtering, terrain-artifact validation or damage inference. See research/visual_review.md."}
for filename,key,row in [("datasets.csv","dataset_id",source_row),("processing_runs.csv","run_id",run_row)]:
    path=ROOT/"metadata"/filename
    with path.open(newline="") as handle:
        reader=csv.DictReader(handle); fields,rows=reader.fieldnames,list(reader)
    previous=next((r for r in rows if r[key]==row[key]),None)
    if filename=="datasets.csv" and previous and previous["sha256"]!=row["sha256"]:
        raise ValueError("Source snapshot changed: use a new dataset ID")
    with path.open("w",newline="") as handle:
        writer=csv.DictWriter(handle,fieldnames=fields,lineterminator="\n");writer.writeheader();writer.writerows([r for r in rows if r[key]!=row[key]]+[row])
print("Registered visual review:",run_id)


## Qualitative review

Read `research/visual_review.md` and inspect both figure PNGs. The April 13 scene is cloud-obscured. Ground patterns are visible in March 29/April 28 candidates but the later view cannot rule out earlier transient flooding. SAR images retain speckle and bright/dark urban/vegetation texture; no unique building flood signal is established.

Only a small number of optical grid centers fall within these outlines. Pixel-center diagnostics depend on the downloaded grid and are not equivalent to earlier weighted-area counts. SAR change exploration would need stable controls, explicit filtering/masking choices, and independent flood evidence. No automatic water or damage labels should follow from this notebook.